In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
from collections import defaultdict
from operator import itemgetter
from typing import Any, Dict, List, Tuple, Optional
from typing import Final
from __future__ import annotations
from enum import Enum
from contextlib import contextmanager

import numpy as np
import pandas as pd
import math
import logging
import os
import pickle
import gpytorch
import torch

In [3]:
logger = logging.getLogger(__name__)

In [4]:
from MRO_library import *
import plots
from radp_library import (
    lon_lat_to_bing_tile,
    bing_tile_to_center_df_row,
    bing_tile_to_center,
    y_to_latitude,
    lon_lat_to_bing_tile_df_row,
    map_clip,
    longitude_to_world_pixel,
    latitude_to_world_pixel,
    get_percell_data
    )

from radp.common.helpers.file_system_safety import (
    atomic_write
)
from radp.digital_twin.utils.gis_tools import GISTools
from radp.digital_twin.rf.bayesian.bayesian_engine import (
    ExactGPModel,
    BayesianDigitalTwin
)

### Helper Functions

### Data Preprocessing

In [5]:
ue_data = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
ue_data

,mock_ue_id,longitude,latitude,tick
0,0,-22.625309,59.806764,0
1,1,119.764151,54.857584,0
2,2,72.095437,-20.253892,0
3,3,-67.548009,-38.100941,0
4,4,59.867089,-83.103930,0
...,...,...,...,...
1995,15,45.564260,42.609846,99
1996,16,132.457280,17.241235,99
1997,17,-101.217659,72.295988,99
1998,18,-16.480045,-26.656397,99


In [6]:
topology = pd.read_csv('data/sim_data/topology.csv')

topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90

topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180

topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 1500
topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2100
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800

topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,1500
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2800


In [7]:
data = concatenate_ue_to_topology(ue_data, topology)
data = connect_ue_to_all_cells(data, topology)

In [8]:
# Calculating distance and received power
data['distance_km'] = data.apply(lambda row: GISTools.get_log_distance(
    row['latitude'], row['longitude'], row['cell_lat'], row['cell_lon']), axis=1)
data['cell_rxpower_dbm'] = data.apply(lambda row: calculate_received_power(
    row['distance_km'], row['cell_carrier_freq_mhz']), axis=1)
# Rename Mock UE ID to UE ID for consistency
data = data.rename(columns={'mock_ue_id': 'ue_id'})
topology["cell_id"] = topology["cell_id"].str.replace('cell_', '').astype(int)
data['cell_id'] = data['cell_id'].str.extract('(\d+)').astype(int)
data = data.rename(columns={'latitude': 'loc_y', 'longitude': 'loc_x'})
data['relative_bearing'] = data.apply(lambda row: GISTools.get_relative_bearing(
    row['cell_az_deg'], row['cell_lat'], row['cell_lon'],row['loc_y'],row['loc_y']), axis=1)
# data = data.rename(columns={'cell_rxpower_dbm': 'rsrp_dbm'})
data = add_sinr_column(data)

In [9]:
data

,ue_id,loc_x,loc_y,tick,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,distance_km,cell_rxpower_dbm,relative_bearing,sinr_db
0,0.0,-22.625309,59.806764,0.0,-90.0,-180.0,1,0,1500,16.629500,-97.389409,239.806764,-19.803317
1,0.0,-22.625309,59.806764,0.0,0.0,0.0,2,120,2100,15.752768,-99.841523,266.698642,-19.803317
2,0.0,-22.625309,59.806764,0.0,90.0,180.0,3,240,2800,15.027772,-101.931053,60.193236,-19.803317
3,1.0,119.764151,54.857584,0.0,-90.0,-180.0,1,0,1500,16.595905,-97.371844,234.857584,-19.646533
4,1.0,119.764151,54.857584,0.0,0.0,0.0,2,120,2100,16.289273,-100.132420,269.925194,-19.646533
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,18.0,-16.480045,-26.656397,99.0,0.0,0.0,2,120,2100,15.054748,-99.447856,101.787576,-14.360112
5996,18.0,-16.480045,-26.656397,99.0,90.0,180.0,3,240,2800,16.379387,-102.679113,146.656397,-14.360112
5997,19.0,34.222834,63.970820,99.0,-90.0,-180.0,1,0,1500,16.656917,-97.403718,243.970820,-16.095630
5998,19.0,34.222834,63.970820,99.0,0.0,0.0,2,120,2100,15.850264,-99.895116,263.693251,-16.095630
